# Poison Detection Agent — Corrected Pipeline

Reordered so it runs **top-to-bottom**. Key fixes vs. the original:

1. **Missing data loader added** — the CSVs are now read into `train_df / test_df / val_df` (the original notebook used them but never created them).
2. **One agent, not two** — the weak 2-signal node and the 4-signal voting class are merged into a single integrated `PoisonDetectionAgent` that computes all signals from the real sample flowing through the graph.
3. **Thresholds wired through** — all four calibrated thresholds are computed *before* the agent is built and are actually passed into it.
4. **KNN statistic made consistent** — calibration and inference both use "mean distance to the 5 nearest *other* points."
5. **Robust embedding-layer reference** — no more hardcoded `"dense_4"`.
6. **Optional label-poisoning path** — a `claimed_label` signal (prediction-label disagreement) is stubbed in, off by default. This is the *only* signal that actually targets **label** poisoning; the other four are out-of-distribution / signal-integrity indicators. Turn it on if incoming samples carry a claimed label.


## 0. Setup

In [ ]:
!pip install -q langgraph langchain-core tensorflow scikit-learn scipy

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import load_model
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import mahalanobis

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Everything lives here. Change if your layout differs.
SPLITS_DIR = "/content/drive/MyDrive/ECG_Splits"
MODEL_PATH = f"{SPLITS_DIR}/DenseNet1D_clean.keras"

# k for the KNN signal (nearest *other* points)
K_NEIGHBORS = 5

Mounted at /content/drive


## 1. Load the diagnosis model

Single `AttentionLayer` definition with `get_config` (needed for deserialization). The duplicate dead definition from the original is gone.

In [ ]:
class AttentionLayer(tf.keras.layers.Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="att_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True,
        )
        self.b = self.add_weight(
            name="att_bias",
            shape=(1,),
            initializer="zeros",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, x):
        scores = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        weights = tf.nn.softmax(scores, axis=1)
        return tf.reduce_sum(x * weights, axis=1)

    def get_config(self):
        return super().get_config()

In [ ]:
model = load_model(
    MODEL_PATH,
    custom_objects={"AttentionLayer": AttentionLayer},
)
print("Model loaded.")

Model loaded.


## 2. Load the data  *(this was missing in the original)*

Assumes each CSV has the 360 signal columns first and an integer `label_enc` column.
Adjust the two lines below if your schema differs.

In [ ]:
train_df = pd.read_csv(f"{SPLITS_DIR}/train.csv")
test_df  = pd.read_csv(f"{SPLITS_DIR}/test.csv")
val_df   = pd.read_csv(f"{SPLITS_DIR}/val.csv")

print("train_df:", train_df.shape)
print("Columns (first few):", list(train_df.columns[:5]), "...", list(train_df.columns[-2:]))

train_df: (69802, 364)
Columns (first few): ['0', '1', '2', '3', '4'] ... ['record_id', 'split']


In [ ]:
# ECG signal (first 360 columns) and encoded labels
X_train = train_df.iloc[:, :360].values.astype(np.float32).reshape(-1, 360, 1)
X_test  = test_df.iloc[:, :360].values.astype(np.float32).reshape(-1, 360, 1)
X_val   = val_df.iloc[:, :360].values.astype(np.float32).reshape(-1, 360, 1)

y_train = train_df["label_enc"].values.astype(np.int32)
y_test  = test_df["label_enc"].values.astype(np.int32)
y_val   = val_df["label_enc"].values.astype(np.int32)

print("X_train:", X_train.shape, "  y_train:", y_train.shape)
print("Classes:", np.unique(y_train))

X_train: (69802, 360, 1)   y_train: (69802,)
Classes: [0 1 2 3]


In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Clean test accuracy: {acc:.4f}")

Clean test accuracy: 0.9179


## 3. Embedding model (robust penultimate-layer reference)

The original hardcoded `get_layer("dense_4")`. Here we default to the **second-to-last layer**
(the feature layer feeding the softmax) and print it so you can confirm — or override
`EMBEDDING_LAYER_NAME` explicitly if you prefer.

In [ ]:
# Inspect layers to confirm the feature layer
for i, layer in enumerate(model.layers):
    shape = getattr(layer, "output_shape", None)
    print(i, layer.name, shape)

0 input_layer_2 None
1 conv1d_30 None
2 batch_normalization_32 None
3 activation_32 None
4 max_pooling1d_2 None
5 batch_normalization_33 None
6 activation_33 None
7 conv1d_31 None
8 concatenate_24 None
9 batch_normalization_34 None
10 activation_34 None
11 conv1d_32 None
12 concatenate_25 None
13 batch_normalization_35 None
14 activation_35 None
15 conv1d_33 None
16 concatenate_26 None
17 batch_normalization_36 None
18 activation_36 None
19 conv1d_34 None
20 concatenate_27 None
21 batch_normalization_37 None
22 activation_37 None
23 conv1d_35 None
24 average_pooling1d_4 None
25 batch_normalization_38 None
26 activation_38 None
27 conv1d_36 None
28 concatenate_28 None
29 batch_normalization_39 None
30 activation_39 None
31 conv1d_37 None
32 concatenate_29 None
33 batch_normalization_40 None
34 activation_40 None
35 conv1d_38 None
36 concatenate_30 None
37 batch_normalization_41 None
38 activation_41 None
39 conv1d_39 None
40 concatenate_31 None
41 batch_normalization_42 None
42 activati

In [ ]:
# Default: penultimate layer. Override with a name if you want a specific one.
EMBEDDING_LAYER_NAME = None  # e.g. "dense_4"

if EMBEDDING_LAYER_NAME:
    feature_layer = model.get_layer(EMBEDDING_LAYER_NAME)
else:
    feature_layer = model.layers[-2]

print("Using embedding layer:", feature_layer.name)

embedding_model = tf.keras.Model(inputs=model.inputs, outputs=feature_layer.output)
print("Embedding output shape:", embedding_model.output_shape)

Using embedding layer: dropout_2
Embedding output shape: (None, 128)


In [ ]:
print("Extracting train embeddings...")
train_embeddings = embedding_model.predict(X_train, batch_size=256, verbose=1)
print("train_embeddings:", train_embeddings.shape)

Extracting train embeddings...
273/273 ━━━━━━━━━━━━━━━━━━━━ 26s 88ms/step
train_embeddings: (69802, 128)


## 4. Calibrate all four thresholds  *(computed before the agent is built)*

Convention for every signal: **`True` = looks clean**. Thresholds are mean ± 3σ over the
clean training set. The KNN statistic is defined identically here and at inference:
mean distance to the `K` nearest *other* points.

In [ ]:
# --- KNN: fit with K+1 so we can drop "self" for train points ---
knn = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1, metric="euclidean")
knn.fit(train_embeddings)

print("Computing train KNN distances...")
train_knn_distances = []
for emb in train_embeddings:
    d, _ = knn.kneighbors(emb.reshape(1, -1))
    # skip index 0 (the point itself) -> mean of K nearest OTHERS
    train_knn_distances.append(np.mean(d[0][1:]))
train_knn_distances = np.array(train_knn_distances)

knn_threshold = train_knn_distances.mean() + 3 * train_knn_distances.std()
print(f"KNN  mean={train_knn_distances.mean():.4f}  "
      f"std={train_knn_distances.std():.4f}  threshold={knn_threshold:.4f}")

Computing train KNN distances...
KNN  mean=0.4562  std=0.3317  threshold=1.4514


In [ ]:
# --- Confidence & Entropy from train softmax outputs ---
print("Computing confidence / entropy distributions...")
train_probs = model.predict(X_train, batch_size=256, verbose=1)

train_conf = np.max(train_probs, axis=1)
train_entropy = -np.sum(train_probs * np.log(train_probs + 1e-10), axis=1)

# clean = HIGH confidence  -> threshold is a LOWER bound
confidence_threshold = train_conf.mean() - 3 * train_conf.std()
# clean = LOW entropy      -> threshold is an UPPER bound
entropy_threshold = train_entropy.mean() + 3 * train_entropy.std()

print(f"Confidence threshold (>=): {confidence_threshold:.4f}")
print(f"Entropy    threshold (<=): {entropy_threshold:.4f}")

Computing confidence / entropy distributions...
273/273 ━━━━━━━━━━━━━━━━━━━━ 25s 87ms/step
Confidence threshold (>=): 0.7223
Entropy    threshold (<=): 0.6314


In [ ]:
# --- Mahalanobis in embedding space ---
print("Computing Mahalanobis statistics...")
maha_mean = np.mean(train_embeddings, axis=0)
cov = np.cov(train_embeddings, rowvar=False)
maha_inv_cov = np.linalg.pinv(cov)

train_maha = np.array([
    mahalanobis(emb, maha_mean, maha_inv_cov) for emb in train_embeddings
])
mahalanobis_threshold = train_maha.mean() + 3 * train_maha.std()
print(f"Mahalanobis mean={train_maha.mean():.4f}  "
      f"std={train_maha.std():.4f}  threshold={mahalanobis_threshold:.4f}")

Computing Mahalanobis statistics...
Mahalanobis mean=10.2600  std=4.7677  threshold=24.5631


## 5. The integrated Poison Detection Agent

One class. Given a single ECG in `state`, it computes **all four signals from that sample**
(no hardcoded numbers), votes, and returns an attack probability. Every signal is an
out-of-distribution / signal-integrity indicator.

The optional **`claimed_label`** signal is the only one that targets **label** poisoning
(prediction-label disagreement). It only activates if the incoming state carries a
non-`None` `claimed_label`, and the denominator adjusts automatically.

In [ ]:
class PoisonDetectionAgent:
    """LangGraph node. Callable on the graph state."""

    def __init__(self, model, embedding_model, knn, maha_mean, maha_inv_cov,
                 confidence_threshold, entropy_threshold,
                 knn_threshold, mahalanobis_threshold,
                 k_neighbors=5, poison_threshold=0.5):
        self.model = model
        self.embedding_model = embedding_model
        self.knn = knn
        self.maha_mean = maha_mean
        self.maha_inv_cov = maha_inv_cov
        self.confidence_threshold = confidence_threshold
        self.entropy_threshold = entropy_threshold
        self.knn_threshold = knn_threshold
        self.mahalanobis_threshold = mahalanobis_threshold
        self.k = k_neighbors
        self.poison_threshold = poison_threshold

    def __call__(self, state):
        ecg = np.expand_dims(state["ecg"], axis=0)

        # --- model outputs ---
        probs = self.model.predict(ecg, verbose=0)[0]
        prediction = int(np.argmax(probs))
        confidence = float(np.max(probs))
        entropy = float(-np.sum(probs * np.log(probs + 1e-10)))

        # --- embedding-space signals ---
        emb = self.embedding_model.predict(ecg, verbose=0)
        d, _ = self.knn.kneighbors(emb)
        knn_distance = float(np.mean(d[0][:self.k]))  # K nearest others
        maha_distance = float(
            mahalanobis(emb.flatten(), self.maha_mean, self.maha_inv_cov)
        )

        # --- evidence: True = looks CLEAN ---
        evidence = {
            "confidence":  confidence   >= self.confidence_threshold,
            "entropy":     entropy      <= self.entropy_threshold,
            "knn":         knn_distance <= self.knn_threshold,
            "mahalanobis": maha_distance <= self.mahalanobis_threshold,
        }

        # --- optional label-poisoning signal (prediction-label disagreement) ---
        claimed_label = state.get("claimed_label", None)
        if claimed_label is not None:
            evidence["label_agreement"] = (prediction == int(claimed_label))

        clean_votes = int(sum(evidence.values()))
        n_signals = len(evidence)
        attack_probability = (n_signals - clean_votes) / n_signals

        return {
            "prediction": prediction,
            "probabilities": probs,
            "confidence": confidence,
            "entropy": entropy,
            "knn_distance": knn_distance,
            "mahalanobis_distance": maha_distance,
            "attack_probability": attack_probability,
            "poison_flag": attack_probability >= self.poison_threshold,
            "evidence": evidence,
        }

## 6. Wire it into LangGraph

The graph now binds the **integrated** agent, fed the calibrated thresholds.

In [ ]:
class ECGState(TypedDict, total=False):
    # Input
    ecg: np.ndarray
    claimed_label: Optional[int]   # set to enable the label-poisoning signal

    # Outputs
    prediction: int
    probabilities: np.ndarray
    confidence: float
    entropy: float
    knn_distance: float
    mahalanobis_distance: float
    attack_probability: float
    poison_flag: bool
    evidence: dict

In [ ]:
poison_agent = PoisonDetectionAgent(
    model=model,
    embedding_model=embedding_model,
    knn=knn,
    maha_mean=maha_mean,
    maha_inv_cov=maha_inv_cov,
    confidence_threshold=confidence_threshold,
    entropy_threshold=entropy_threshold,
    knn_threshold=knn_threshold,
    mahalanobis_threshold=mahalanobis_threshold,
    k_neighbors=K_NEIGHBORS,
)

builder = StateGraph(ECGState)
builder.add_node("PoisonDetection", poison_agent)
builder.add_edge(START, "PoisonDetection")
builder.add_edge("PoisonDetection", END)
graph = builder.compile()
print("Graph compiled.")

Graph compiled.


## 7. Sanity checks

**7a. Clean sample** — should give a low attack probability.

In [ ]:
clean = graph.invoke({"ecg": X_test[25]})
print("prediction:        ", clean["prediction"])
print("confidence:        ", round(clean["confidence"], 4))
print("entropy:           ", round(clean["entropy"], 4))
print("knn_distance:      ", round(clean["knn_distance"], 4))
print("mahalanobis:       ", round(clean["mahalanobis_distance"], 4))
print("attack_probability:", clean["attack_probability"])
print("poison_flag:       ", clean["poison_flag"])
print("evidence:          ", clean["evidence"])

prediction:         2
confidence:         0.9995
entropy:            0.0047
knn_distance:       0.8229
mahalanobis:        12.9296
attack_probability: 0.0
poison_flag:        False
evidence:           {'confidence': np.True_, 'entropy': np.True_, 'knn': np.True_, 'mahalanobis': np.True_}


**7b. Signal-perturbed sample** — inject strong noise. The OOD / signal-integrity signals
(KNN, Mahalanobis, confidence, entropy) should push the attack probability up. This
demonstrates detection of a **signal-level** perturbation.

In [ ]:
rng = np.random.default_rng(0)
perturbed = X_test[25] + rng.normal(0, 0.5, size=X_test[25].shape).astype(np.float32)

res = graph.invoke({"ecg": perturbed})
print("attack_probability:", res["attack_probability"])
print("poison_flag:       ", res["poison_flag"])
print("evidence:          ", res["evidence"])

attack_probability: 0.5
poison_flag:        True
evidence:           {'confidence': np.True_, 'entropy': np.True_, 'knn': np.False_, 'mahalanobis': np.False_}


**7c. Label-poisoning path** — pass a *wrong* `claimed_label`. The `label_agreement`
signal fires (prediction-label disagreement) even though the signal itself is clean. This
is the branch that actually addresses **label** poisoning, per the agent spec.

In [ ]:
true_label = int(y_test[25])
wrong_label = (true_label + 1) % 4  # deliberately mismatched

res = graph.invoke({"ecg": X_test[25], "claimed_label": wrong_label})
print("true label:        ", true_label)
print("claimed label:     ", wrong_label)
print("prediction:        ", res["prediction"])
print("attack_probability:", res["attack_probability"])
print("poison_flag:       ", res["poison_flag"])
print("evidence:          ", res["evidence"])

true label:         2
claimed label:      3
prediction:         2
attack_probability: 0.2
poison_flag:        False
evidence:           {'confidence': np.True_, 'entropy': np.True_, 'knn': np.True_, 'mahalanobis': np.True_, 'label_agreement': False}
